In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.discrete.conditional_models import ConditionalLogit
import re

In [2]:
file_path = "问卷数据400份2.19.xlsx"
print("加载数据。")
try:
    df_raw = pd.read_excel(file_path)
    print(f"原始样本量: {len(df_raw)}")
except FileNotFoundError:
    print("错误：找不到文件。")

加载数据。
原始样本量: 399


In [3]:
trap_cols = [c for c in df_raw.columns if 'Q5' in c and '显示测试' in c]
if not trap_cols:
    print("错误：未找到陷阱题列。")
    valid_df = df_raw.copy()
else:
    trap_col = trap_cols[0]
    eliminated_mask = (df_raw[trap_col] == 1) | (df_raw[trap_col] == 1.0)
    valid_df = df_raw[~eliminated_mask].copy()
    print(f"保留样本人数: {len(valid_df)}")

保留样本人数: 285


In [4]:
def parse_attributes_safe(text, is_none=False):
    if is_none:
        return {'Smart': 0, 'Context': 0, 'Privacy': 0, 'Price': 0, 'Buy': 0}
    
    text = str(text).replace('：', ':').replace(' ', '')
    segments = re.split(r'【|】', text)
    attrs = {'Smart': 1, 'Context': 1, 'Privacy': 1, 'Price': 0, 'Buy': 1}
    
    for i in range(len(segments)-1):
        key = segments[i]
        val = segments[i+1]
        if '智能' in key:
            if 'LV3' in val or '专家' in val: attrs['Smart'] = 3
            elif 'LV2' in val or '进阶' in val: attrs['Smart'] = 2
        elif '上下文' in key:
            if 'LV3' in val or '超长' in val: attrs['Context'] = 3
            elif 'LV2' in val or '较长' in val: attrs['Context'] = 2
        elif '隐私' in key:
            if 'LV2' in val or '严格保密' in val: attrs['Privacy'] = 2
        elif '价格' in key:
            nums = re.findall(r'(\d+)', val)
            if nums: attrs['Price'] = int(nums[0])
    return attrs

long_data = []
choice_cols = [c for c in df_raw.columns if '方案A' in c and 'Q5' not in c]

for idx, row in valid_df.iterrows():
    respondent_id = row.get('序号', idx)
    for q_col in choice_cols:
        try:
            parts = str(q_col).split('方案B')
            text_a = parts[0].split('方案A')[-1]
            text_b = parts[1]
        except: 
            continue
        
        user_val = row[q_col]
        options = [('A', text_a, False), ('B', text_b, False), ('None', '', True)]
        
        for label, text, is_none_flag in options:
            attrs = parse_attributes_safe(text, is_none=is_none_flag)
            is_chosen = 0
            
            if label == 'A' and user_val == 1: is_chosen = 1
            elif label == 'B' and user_val == 2: is_chosen = 1
            elif label == 'None' and user_val == 3: is_chosen = 1
            
            long_data.append({'ID': respondent_id, 'Q_ID': q_col[:10], 'Choice': is_chosen, **attrs})

df_long = pd.DataFrame(long_data)
print("数据重构完成。")

数据重构完成。


In [5]:
df_model = df_long.copy()
for attr in ['Smart', 'Context', 'Privacy']:
    levels = sorted(df_model[attr].unique())
    for lv in levels[1:]:
        if lv > 0:
            df_model[f'{attr}_{lv}'] = (df_model[attr] == lv).astype(int)

df_model = df_model.sort_values(by=['ID', 'Q_ID'])
df_model['Task_Group'] = df_model['ID'].astype(str) + "_" + df_model['Q_ID'].astype(str)

X_cols = ['Buy', 'Price', 'Smart_2', 'Smart_3', 'Context_2', 'Context_3', 'Privacy_2']
y = df_model['Choice']
X = df_model[X_cols]
groups = df_model['Task_Group']

print("运行条件Logit模型。")
try:
    clogit_model = ConditionalLogit(y, X, groups=groups)
    result = clogit_model.fit(disp=0)
    print(result.summary())
    
    beta_price = result.params['Price']
    print("计算支付意愿。")
    for col in [c for c in X_cols if c not in ['Buy', 'Price']]:
        wtp = - (result.params[col] / beta_price)
        print(f"[{col}] 对比基准的 WTP: {wtp:.2f} 元/月")
        
except Exception as e:
    print(f"模型运行失败: {e}")

运行条件Logit模型。


C:\Users\liuyu\AppData\Roaming\Python\Python312\site-packages\statsmodels\discrete\conditional_models.py:80: UserWarning: Dropped 9 groups and 27 observations for having no within-group variance
  warnings.warn(msg)


                  Conditional Logit Model Regression Results                  
Dep. Variable:                 Choice   No. Observations:                 7668
Model:               ConditionalLogit   No. groups:                       1728
Log-Likelihood:               -3172.1   Min group size:                      3
Method:                          BFGS   Max group size:                      6
Date:                Thu, 19 Feb 2026   Mean group size:                   4.4
Time:                        22:59:07                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Buy            0.3026      0.106      2.848      0.004       0.094       0.511
Price         -0.0038      0.001     -4.122      0.000      -0.006      -0.002
Smart_2        0.3399      0.097      3.489      0.000       0.149       0.531
Smart_3        0.1624      0.100      1.628      0.1